# Stage 1 — 도시 장애물 환경 학습 (로컬 GPU)

**담당**: 이재왕 (work/evader)  
**씬**: `Assets/01. Scenes/Stage1.unity`  
**목표**: 도시 환경에서 장애물 회피 + GoalZone 도달 학습 (Pursuer 없음)  
**전략**: Stage1-A (`_goalOnlyMode=true`, `_currentStage=1`)  
**수렴 기준**: `goal_reach_rate ≥ 30%`, `crash_rate ≤ 15%` → Stage1-B (RL Pursuer 추가)  

---

## 사전 준비 (최초 1회)

```bash
cd c:\IIT_DroneLearning
.venv\Scripts\activate
jupyter notebook python/notebooks/stage1_obstacle_local.ipynb
```

---

---
## 1. 환경 확인

In [11]:
import sys
import torch
import mlagents_envs

print(f'Python   : {sys.version.split()[0]}')
print(f'torch    : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'mlagents : {mlagents_envs.__version__}')


Python   : 3.10.11
torch    : 2.7.0+cu128
CUDA     : True
GPU      : NVIDIA GeForce RTX 5060
VRAM     : 8.5 GB
mlagents : 1.2.0.dev0


---
## 2. 경로 및 실험 설정

In [ ]:
from pathlib import Path

REPO_PATH  = Path("c:/IIT_DroneLearning")
LOG_DIR    = REPO_PATH / "python" / "results"
CONFIG_DIR = REPO_PATH / "python" / "config"

LOG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

# ===== Next-task profile =====
# "eval_v6"    : 학습 실행 없이 v6 결과 점검/평가
# "train_v7"   : v6를 초기값으로 v7 새 학습 시작 (--force + --initialize-from)
# "resume_v7"  : 기존 v7 이어학습 (--resume)
WORKFLOW_MODE = "train_v7"

PREV_RUN_ID = f"evader_s1_obstacle_44d_v6_seed{SEED}"
V7_RUN_ID   = f"evader_s1_obstacle_44d_v7_seed{SEED}"

if WORKFLOW_MODE == "eval_v6":
    INIT_FROM = PREV_RUN_ID
    RUN_ID = PREV_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = None
elif WORKFLOW_MODE == "train_v7":
    INIT_FROM = PREV_RUN_ID
    RUN_ID = V7_RUN_ID
    RESUME = False
    TARGET_STOP_STEP = 500000
elif WORKFLOW_MODE == "resume_v7":
    INIT_FROM = ""
    RUN_ID = V7_RUN_ID
    RESUME = True
    TARGET_STOP_STEP = 500000
else:
    raise ValueError(f"Unknown WORKFLOW_MODE: {WORKFLOW_MODE}")

EVAL_RUN_ID = PREV_RUN_ID if WORKFLOW_MODE == "eval_v6" else RUN_ID
EVAL_EPISODES = 50
EVAL_SEED = 42

print(f"Repo          : {REPO_PATH}")
print(f"Log dir       : {LOG_DIR}")
print(f"Workflow mode : {WORKFLOW_MODE}")
print(f"Run ID        : {RUN_ID}")
print(f"Init From     : {INIT_FROM or '(none)'}")
print(f"Resume        : {RESUME}")
print(f"Target step   : {TARGET_STOP_STEP}")
print(f"Eval run-id   : {EVAL_RUN_ID} / episodes={EVAL_EPISODES} / seed={EVAL_SEED}")


Repo          : c:\IIT_DroneLearning
Log dir       : c:\IIT_DroneLearning\python\results
Workflow mode : eval_v6
Run ID        : evader_s1_obstacle_44d_v6_seed42
Init From     : evader_s1_obstacle_44d_v6_seed42
Resume        : False
Target step   : None
Eval run-id   : evader_s1_obstacle_44d_v6_seed42 / episodes=50 / seed=42


---
## 3. Config 확인

In [26]:
config_path = CONFIG_DIR / 'evader_s1_obstacle_template.yaml'
assert config_path.exists(), f'Config 없음: {config_path}'

print(f'Config: {config_path}{"=" * 60}')
print(config_path.read_text(encoding='utf-8'))


Config: c:\IIT_DroneLearning\python\config\evader_s1_obstacle_template.yaml============================================================
# evader_s1_obstacle_template.yaml
# Stage1-A v5: Resume from v1-500k (has wall avoidance) + Goal on Ignore Raycast
# Goal: Mean Reward > +1.0 within 200k steps
#
# WHY v5: v4 (Stage0 warm-start) had zero obstacle knowledge -> constant crashes.
# v1-500k knows wall avoidance AND some goal-reaching (peak +2.669 at 340k).
# CRITICAL FIX: Goal GameObject moved to "Ignore Raycast" layer ->
#   proximity penalty no longer fires on Goal collider -> goal-avoidance bias gone.
#
# CURRICULUM:
#   Phase1 (v5): SpawnCenter radius=12m  -> max spawn-goal dist ~24m
#   Phase2 (v6): SpawnCenter radius=25m  -> after Phase1 converges
#   Phase3 (v7): full city range (CityMetadata strategy)
#
# Unity Inspector (Stage1.unity):
#   EpisodeSpawnCoordinator:
#     Strategy        = SpawnCenterRandom
#     _minSeparation  = 3
#   SpawnCenter:
#     Auto Sync From City = OFF


---
## 4. Unity Editor 연결 확인

⚠️ Unity에서 `Stage1.unity` 씬을 열고 Inspector 값을 먼저 확인하세요.  
`WORKFLOW_MODE = "eval_v6"` 일 때는 5번 학습 셀을 실행하지 않습니다.  
학습을 재시작할 때만 `WORKFLOW_MODE = "train_v7"` 또는 `"resume_v7"`로 변경하세요.

**Unity Inspector 체크리스트 (v7 hardening 반영):**

| 컴포넌트 | 필드 | 값 |
|---|---|---|
| **Goal** | **Layer** | **Ignore Raycast** (필수) |
| EpisodeSpawnCoordinator | Strategy | SpawnCenterRandom |
| SpawnCenter | Radius | 12 |
| EvaderAgent | _currentStage | 1 |
| EvaderAgent | _goalOnlyMode | true |
| EvaderAgent | Max Episode Seconds | 40 |
| EvaderAgent | _goalArrivalReward | 3.0 |
| EvaderAgent | _checkpointCount | 2 |
| EvaderAgent | _checkpointRadius | 4.0 |
| EvaderAgent | _checkpointReward | 0.10 |
| EvaderAgent | _autoSetGoalIgnoreRaycastLayer | true |
| EvaderAgent | _excludeIgnoreRaycastFromSensorMask | true |
| EvaderAgent | _disableMiddleBottomRaysInStage1 | true |
| EvaderAgent | _disableBottomRayInStage1 | true |
| EvaderReward | _goalShapingCoeff | 0.3 |
| EvaderReward | _velAlignCoeff | 0.012 |
| EvaderReward | _goalPriorityDist | 6.0 |
| EvaderReward | _timePenaltyPerStep | -0.001 |
| EvaderReward | _proximityCoeff | 0.006 |
| EvaderReward | _proximityThreshold | 0.25 |
| EvaderReward | _nearGoalObstaclePenaltyScale | 0.35 |
| EvaderReward | _middleBottomRayPenaltyWeight | 0.15 |
| EvaderReward | _bottomRayPenaltyWeight | 0.0 |
| BehaviorParameters | Behavior Name | Drone_Evader |
| BehaviorParameters | Behavior Type | Default |

In [27]:
import socket

port = 5004
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    result = s.connect_ex(('127.0.0.1', port))
    if result == 0:
        print(f'⚠️  포트 {port} 이미 사용 중 — 이전 mlagents-learn 프로세스가 남아있을 수 있습니다.')
    else:
        print(f'✅ 포트 {port} 사용 가능. 다음 셀 실행 후 Unity에서 Play 하세요.')


✅ 포트 5004 사용 가능. 다음 셀 실행 후 Unity에서 Play 하세요.


---
## 5. 학습 실행 (train/resume 모드 전용)

이 셀은 `WORKFLOW_MODE`가 `train_v7` 또는 `resume_v7`일 때만 실행하세요.  
`eval_v6` 모드에서는 셀이 자동으로 skip 됩니다.

실행 순서:
1. 이 셀 실행 (mlagents-learn 대기)
2. Unity Editor -> Stage1.unity -> Play
3. target step 도달 시 자동 중단

In [28]:
import subprocess, sys, socket, time, threading, json, os, re
from pathlib import Path

mode = globals().get("WORKFLOW_MODE", "train_v7")
if mode == "eval_v6":
    print("[skip] WORKFLOW_MODE='eval_v6' 입니다. 학습 셀을 건너뜁니다.")
    print("      학습을 하려면 설정 셀에서 WORKFLOW_MODE를 'train_v7' 또는 'resume_v7'로 바꾼 뒤 다시 실행하세요.")
    raise SystemExit(0)

venv_scripts = Path(sys.executable).parent
mlagents_bin = venv_scripts / 'mlagents-learn.exe'
BASE_PORT = 5004
RETRY_ON_PORT_IN_USE = True

# RESUME=True: --resume (이어받기) / False: --force + warm-start
cmd = [
    str(mlagents_bin), str(config_path),
    f'--run-id={RUN_ID}',
    f'--results-dir={LOG_DIR}',
]
if RESUME:
    cmd += ['--resume']
    print('Mode: RESUME — 마지막 체크포인트에서 이어받습니다.')
else:
    cmd += ['--force']
    if INIT_FROM:
        cmd += [f'--initialize-from={INIT_FROM}']
        print(f'Mode: FORCE — Warm-start from: {INIT_FROM}')
    else:
        print('Mode: FORCE — 처음부터 학습합니다.')

# 목표 스텝 도달 시 안전 중단 (None이면 비활성)
if 'TARGET_STOP_STEP' not in globals():
    TARGET_STOP_STEP = 500000
POLL_INTERVAL_SEC = 2
MONITOR_HEARTBEAT_SEC = 30
MONITOR_STALE_WARN_SEC = 120

print('실행 커맨드:')
print(' '.join(str(c) for c in cmd))
print()
if TARGET_STOP_STEP is not None:
    print(f'Auto-stop target step: {TARGET_STOP_STEP}')

output_lines = []
latest_step = {'value': None}

# Example lines:
# [INFO] Drone_Evader. Step: 360000. Time Elapsed: ...
# [INFO] Resuming training from step 353848.
step_pattern = re.compile(r"Step:\s*(\d+)")
resume_pattern = re.compile(r"Resuming training from step\s*(\d+)")
onnx_pattern = re.compile(r"-(\d+)\.onnx$")


def kill_stale_mlagents_processes():
    if os.name != 'nt':
        return
    try:
        subprocess.run(
            ['taskkill', '/F', '/IM', 'mlagents-learn.exe', '/T'],
            capture_output=True,
            text=True,
            check=False,
        )
    except Exception as e:
        print(f'[warn] stale process cleanup failed: {e}')


def port_is_busy(port_number: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port_number)) == 0


def stream_output(pipe):
    for line in pipe:
        text = line.decode('utf-8', errors='replace').rstrip()
        output_lines.append(text)

        m_step = step_pattern.search(text)
        if m_step:
            try:
                latest_step['value'] = int(m_step.group(1))
            except ValueError:
                pass

        m_resume = resume_pattern.search(text)
        if m_resume:
            try:
                latest_step['value'] = int(m_resume.group(1))
            except ValueError:
                pass

        # Exported ... Drone_Evader-349884.onnx 형태도 스텝으로 반영
        if 'Exported ' in text and '.onnx' in text:
            try:
                tail = text.split()[-1]
                m_onnx = onnx_pattern.search(tail)
                if m_onnx:
                    latest_step['value'] = int(m_onnx.group(1))
            except Exception:
                pass

        print(text)


def read_step_snapshot(log_root: Path, run_id: str):
    """run_logs에서 가능한 최신 step 후보를 읽고 최대값을 반환한다."""
    run_dir = log_root / run_id / 'run_logs'
    timers_path = run_dir / 'timers.json'
    status_path = run_dir / 'training_status.json'

    candidates = []

    if timers_path.exists():
        try:
            timers = json.loads(timers_path.read_text(encoding='utf-8'))
            gauges = timers.get('gauges', {})
            step_keys = [k for k in gauges.keys() if k.endswith('.Step.mean')]
            if step_keys:
                val = gauges[step_keys[0]].get('value', None)
                if isinstance(val, (int, float)):
                    candidates.append(int(val))
        except Exception:
            pass

    if status_path.exists():
        try:
            status = json.loads(status_path.read_text(encoding='utf-8'))
            for key, val in status.items():
                if key == 'metadata' or not isinstance(val, dict):
                    continue
                final_ckpt = val.get('final_checkpoint', {})
                step = final_ckpt.get('steps', None)
                if isinstance(step, int):
                    candidates.append(step)
        except Exception:
            pass

    if not candidates:
        return None
    return max(candidates)


# 실행 전 pre-check: 이미 target 이상이면 프로세스를 시작하지 않는다.
pre_step = read_step_snapshot(LOG_DIR, RUN_ID)
if TARGET_STOP_STEP is not None and pre_step is not None:
    print(f'[pre-check] snapshot_step={pre_step} / target={TARGET_STOP_STEP}')
    if pre_step >= TARGET_STOP_STEP:
        print()
        print('학습 시작 스킵: 현재 체크포인트가 이미 target step 이상입니다.')
        print(f'  - current_step={pre_step}, target={TARGET_STOP_STEP}')
        print('  - 더 학습하려면 TARGET_STOP_STEP을 더 크게 설정하세요. (예: 600000)')
        raise SystemExit(0)

if port_is_busy(BASE_PORT):
    print(f'[pre-check] port {BASE_PORT} is busy -> stale mlagents-learn cleanup 시도')
    kill_stale_mlagents_processes()
    time.sleep(2)
    if port_is_busy(BASE_PORT):
        print(f'[pre-check] port {BASE_PORT} still busy after cleanup. Unity Play 또는 이전 세션을 확인하세요.')

print(f'⏳ mlagents-learn 시작 중... (포트 {BASE_PORT} 준비 대기)')

creationflags = 0
if os.name == 'nt' and hasattr(subprocess, 'CREATE_NEW_PROCESS_GROUP'):
    creationflags = subprocess.CREATE_NEW_PROCESS_GROUP

attempt = 0
max_attempts = 2
while attempt < max_attempts:
    if attempt > 0:
        print(f'[retry] mlagents-learn 재시도 {attempt + 1}/{max_attempts}')

    output_lines.clear()
    latest_step['value'] = None

    proc = subprocess.Popen(
        cmd, cwd=str(REPO_PATH),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        creationflags=creationflags
    )
    t = threading.Thread(target=stream_output, args=(proc.stdout,), daemon=True)
    t.start()

    port = BASE_PORT
    ready = False
    for _ in range(60):
        time.sleep(0.5)
        if proc.poll() is not None:
            t.join(timeout=2)
            break
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.3)
            if s.connect_ex(('127.0.0.1', port)) == 0:
                ready = True
                break

    if not ready and proc.poll() is not None:
        captured_output = '\n'.join(output_lines)
        if RETRY_ON_PORT_IN_USE and attempt == 0 and ('UnityWorkerInUseException' in captured_output or 'Failed to bind to address' in captured_output):
            print('[retry] worker/port in use detected. stale process cleanup 후 재시도합니다.')
            kill_stale_mlagents_processes()
            time.sleep(2)
            attempt += 1
            continue
        print(f'\n❌ mlagents-learn이 즉시 종료되었습니다 (exit={proc.returncode})')
        print(captured_output)
        raise RuntimeError('mlagents-learn 시작 실패 — 위 출력 내용을 확인하세요.')

    if ready:
        print()
        print('=' * 55)
        print(f'✅ mlagents-learn 준비 완료 — 포트 {BASE_PORT} listening')
        print('▶  지금 Unity Editor에서 Stage1.unity → Play 누르세요!')
        print('=' * 55)

        stop_requested = False
        last_monitor_step = None
        last_step_change_ts = time.time()
        last_heartbeat_print_ts = 0.0
        last_stale_warn_ts = 0.0

        while proc.poll() is None:
            time.sleep(POLL_INTERVAL_SEC)
            if TARGET_STOP_STEP is None:
                continue

            file_step = read_step_snapshot(LOG_DIR, RUN_ID)
            current_step = latest_step['value']
            if current_step is None:
                current_step = file_step
            elif file_step is not None:
                current_step = max(current_step, file_step)

            if current_step is not None:
                now_ts = time.time()

                if last_monitor_step is None or current_step != last_monitor_step:
                    last_monitor_step = current_step
                    last_step_change_ts = now_ts
                    last_heartbeat_print_ts = now_ts
                    print(f'[monitor:step-change] current_step={current_step} / target={TARGET_STOP_STEP}')
                elif now_ts - last_heartbeat_print_ts >= MONITOR_HEARTBEAT_SEC:
                    last_heartbeat_print_ts = now_ts
                    print(f'[monitor:heartbeat] current_step={current_step} / target={TARGET_STOP_STEP}')

                if (
                    now_ts - last_step_change_ts >= MONITOR_STALE_WARN_SEC
                    and now_ts - last_stale_warn_ts >= MONITOR_HEARTBEAT_SEC
                ):
                    last_stale_warn_ts = now_ts
                    print('[monitor:stale] step이 일정 시간 변하지 않았습니다. Unity Play 상태/환경 속도/학습 로그를 확인하세요.')

                if current_step >= TARGET_STOP_STEP and not stop_requested:
                    print(f'\n🛑 target step 도달 ({current_step} >= {TARGET_STOP_STEP}), 학습 프로세스를 중단합니다...')
                    stop_requested = True
                    try:
                        proc.terminate()
                    except Exception as e:
                        print(f'[warn] terminate 실패: {e}; kill로 전환')
                        try:
                            proc.kill()
                        except Exception as e2:
                            print(f'[warn] kill 실패: {e2}')

        t.join(timeout=5)
        if TARGET_STOP_STEP is not None and stop_requested:
            if proc.returncode not in (0, None):
                print(f'\n학습 종료 (exit={proc.returncode}) - target auto-stop에 따른 종료로 처리합니다.')
            else:
                print(f'\n학습 종료 (exit={proc.returncode})')
            print('다음 실행은 RESUME=True로 같은 RUN_ID를 사용하세요.')
        else:
            print(f'\n학습 종료 (exit={proc.returncode})')
        break

    attempt += 1

else:
    raise RuntimeError('mlagents-learn 시작 실패 — 위 출력 내용을 확인하세요.')

[skip] WORKFLOW_MODE='eval_v6' 입니다. 학습 셀을 건너뜁니다.
      학습을 하려면 설정 셀에서 WORKFLOW_MODE를 'train_v7' 또는 'resume_v7'로 바꾼 뒤 다시 실행하세요.


SystemExit: 0

---
## 6. TensorBoard 모니터링

학습 중 **새 터미널**에서 아래 명령어로 TensorBoard를 실행하세요.


In [ ]:
tb_cmd = f'.venv\\Scripts\\tensorboard --logdir python/results/{RUN_ID} --port 6006'
print('새 터미널에서 실행:')
print(tb_cmd)
print()
print('브라우저: http://localhost:6006')
print()
print('주요 모니터링 지표:')
print('  Environment/Cumulative Reward  → 상승 추세 확인 (초반 1~3 기대)')
print('  Environment/Episode Length     → 500~1000 (탐색 중 정상)')
print('  Policy/Entropy                 → 서서히 감소해야 함')
print()
print('⚠️  위험 신호:')
print('  - 50k 스텝 이후에도 Mean Reward 음수 지속 → 보상 재설계 필요')
print('  - Episode Length 항상 Max(1250) → 타임아웃 과다 (crash 또는 방황)')


---
## 7. 결과 확인 및 ONNX 경로

In [29]:
run_dir = LOG_DIR / RUN_ID

if run_dir.exists():
    onnx_files = list(run_dir.glob('**/*.onnx'))
    pt_files   = list(run_dir.glob('**/*.pt'))

    print(f'✅ 결과 폴더: {run_dir}')
    print(f'ONNX 파일 ({len(onnx_files)}개):')
    for f in sorted(onnx_files):
        print(f'  {f.name}')
    print(f'체크포인트 ({len(pt_files)}개): {len(pt_files)}개')
else:
    print(f'❌ 결과 폴더 없음: {run_dir}')
    print('5번 셀(학습)을 먼저 실행하세요.')


✅ 결과 폴더: c:\IIT_DroneLearning\python\results\evader_s1_obstacle_44d_v6_seed42
ONNX 파일 (7개):
  Drone_Evader-149932.onnx
  Drone_Evader-199927.onnx
  Drone_Evader-249962.onnx
  Drone_Evader-299940.onnx
  Drone_Evader-303555.onnx
  Drone_Evader-49958.onnx
  Drone_Evader.onnx
체크포인트 (7개): 7개


---
## 8. Eval 실행 (고정 시드)

1. Unity에서 평가 모드로 `EVAL_RUN_ID` 모델을 로드해 에피소드 로그(`episodes.csv` 또는 `episodes.ndjson`)를 생성합니다.
2. 아래 셀을 실행해 지표를 계산하고 `metrics.json`을 저장합니다.
3. 결과를 `docs/EXPERIMENTS.md`에 기록합니다.

In [30]:
import subprocess, sys

eval_script = REPO_PATH / "python" / "scripts" / "eval.py"
assert eval_script.exists(), f"eval.py 없음: {eval_script}"

eval_cmd = [
    sys.executable,
    str(eval_script),
    "--run-id", EVAL_RUN_ID,
    "--n-episodes", str(EVAL_EPISODES),
    "--seed", str(EVAL_SEED),
]

print("Eval command:")
print(" ".join(eval_cmd))
print()

result = subprocess.run(eval_cmd, cwd=str(REPO_PATH), text=True)
if result.returncode != 0:
    print()
    print("[hint] episodes.csv/episodes.ndjson가 없으면 Unity 평가 실행 후 로그를 먼저 export 하세요.")

Eval command:
c:\IIT_DroneLearning\.venv\Scripts\python.exe c:\IIT_DroneLearning\python\scripts\eval.py --run-id evader_s1_obstacle_44d_v6_seed42 --n-episodes 50 --seed 42


[hint] episodes.csv/episodes.ndjson가 없으면 Unity 평가 실행 후 로그를 먼저 export 하세요.


---
## 9. Stage1-B 전환 (수렴 확인 후)

수렴 기준:
- `goal_reach_rate ≥ 30%`
- `crash_rate ≤ 15%`
- 최근 100k 구간에서 mean reward 급락 없음

**Unity Inspector 변경 (Stage1-B):**
- `EvaderAgent._goalOnlyMode = false` (RL Pursuer 활성)
- Pursuer 오브젝트 -> BehaviorParameters -> Model = `pursuer_s2_catch_v3_499980.onnx`

**다음 버전 설정 예시:**
```python
# Stage1-B는 Stage1-A 최종 run을 초기값으로 사용
INIT_FROM = f"evader_s1_obstacle_44d_v7_seed{SEED}"  # 또는 best obstacle run-id
RUN_ID    = f"evader_s1_pursuer_44d_v1_seed{SEED}"
RESUME    = False
TARGET_STOP_STEP = 300000
```
